# Parse → chunk → search, one step at a time

**A personal hobby project**, not production code and not affiliated with turbopuffer. This notebook runs the same pipeline as `app.py`, but step by step so each stage can be inspected. The sample document is a fictional employee handbook.

You need `TURBOPUFFER_API_KEY` set for steps 4 and 5, and optionally `ANTHROPIC_API_KEY` for step 6.

In [ ]:
import os
from pathlib import Path
from pipeline import parse, chunk, search

pdf = Path("samples/harbor-lane-handbook.pdf").read_bytes()
doc_id = search.doc_id_for("harbor-lane-handbook.pdf", pdf)
doc_id

## 1. Parse
First the naive way, then a structured parser. Notice the running header, the hard line wraps and the flattened table in the raw version.

In [ ]:
raw, pages = parse.raw_text(pdf)
print(raw[:1200])

In [ ]:
parsed = parse.parse(pdf, "pymupdf")   # or "docling" if you installed requirements-docling.txt
print("Removed as header/footer:", parsed["removed"])
print(parsed["markdown"][:1500])

## 2. Sections
The heading path is what keeps a chunk's context once it's cut out of the document.

In [ ]:
for s in chunk.split_sections(parsed["markdown"]):
    print(f"{s.path:45} {len(s.body):5} chars")

## 3. Chunk two ways

In [ ]:
rec = chunk.recursive_chunks(parsed["markdown"], chunk_chars=800, overlap_chars=100)
sem, gaps = chunk.semantic_chunks(parsed["markdown"], breakpoint_percentile=20, max_chars=1500)
print("recursive:", chunk.summary(rec))
print("semantic: ", chunk.summary(sem))

In [ ]:
# Where did semantic chunking cut, and why?
import pandas as pd
pd.DataFrame(gaps)[["similarity", "cut", "reason", "cross_section", "after", "before"]]

In [ ]:
# Chunks that cross a heading boundary: recursive never does this, semantic sometimes does
for c in sem:
    if len(c.sections) > 1:
        print(c.sections, "\n   ", c.text[:150], "\n")

## 4. Store in turbopuffer
One namespace per strategy. turbopuffer embeds `content` on write (native embedding) and builds a BM25 index on it.

In [ ]:
tpuf = search.client()
for name, chunks in {"recursive": rec, "semantic": sem}.items():
    rows = [c.to_row(doc_id) for c in chunks]
    print(search.upload(tpuf, doc_id, name, rows))

## 5. Search four ways

In [ ]:
q = "Who has to sign off on a $700 expense?"
for mode in search.MODES:
    r = search.search(tpuf, doc_id, "recursive", q, mode, top_k=3)
    print(f"--- {mode} ({r['ms']} ms) {r['note']}")
    for row in r["rows"]:
        print(f"  {row['score']}  {row['section']:35}  {row['content'][:90]!r}")

In [ ]:
# Same question, both chunking strategies, hybrid mode
for strat in ("recursive", "semantic"):
    r = search.search(tpuf, doc_id, strat, "How much can I spend on dinner during a work trip to London?", "hybrid", 3)
    print(f"--- {strat}")
    for row in r["rows"]:
        print(f"  {row['section']:35}  {row['content'][:110]!r}")

## 6. Answer with Claude (optional)

In [ ]:
r = search.search(tpuf, doc_id, "recursive", q, "hybrid", top_k=4)
print(search.answer(q, r["rows"]))

## Clean up
Deletes every namespace this project created (they all start with the `tpragdemo-` prefix).

In [ ]:
# search.delete_demo_namespaces(tpuf)